<a href="https://colab.research.google.com/github/N-S-Wickramanayaka/Brahmee-transformer/blob/main/Brahmi_letter_recognizer_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {gpu_props.total_memory / 1e9:.1f} GB")
    print(f"Free Memory: {torch.cuda.mem_get_info(0)[0] / 1e9:.1f} GB")
else:
    print("⚠ No GPU! Go to Runtime → Change runtime type → GPU")

In [ ]:
!pip install -q timm==0.9.12
!pip install -q albumentations==1.3.1
!pip install -q scikit-learn matplotlib seaborn tqdm
!pip install -q kagglehub
print("✅ All packages installed!")

In [ ]:
from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("✅ Kaggle API configured!")

In [ ]:
# Try kagglehub first (newer method)
try:
    import kagglehub
    path = kagglehub.dataset_download("vajirj/sinhala-early-brahmi-inscription-dataset")
    print(f"Downloaded to: {path}")

    import shutil
    target = './brahmi_dataset'
    if os.path.exists(target):
        shutil.rmtree(target)
    shutil.copytree(path, target)
    print(f"✅ Dataset copied to: {target}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    print("Trying old method...")
    !kaggle datasets download -d vajirj/sinhala-early-brahmi-inscription-dataset
    !unzip -q sinhala-early-brahmi-inscription-dataset.zip -d ./brahmi_dataset
    print("✅ Dataset downloaded!")


In [ ]:
import os

dataset_path = './brahmi_dataset'

def find_class_root(path):
    """Find the directory containing class subdirectories."""
    for root, dirs, _ in os.walk(path):
        if len(dirs) > 50:
            sample = os.path.join(root, dirs[0])
            if any(f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))
                   for f in os.listdir(sample)):
                return root
    return path

DATA_ROOT = find_class_root(dataset_path)
print(f"DATA_ROOT: {DATA_ROOT}")

classes = sorted([d for d in os.listdir(DATA_ROOT)
                  if os.path.isdir(os.path.join(DATA_ROOT, d))])
print(f"Total classes: {len(classes)}")

# Count images per class
class_counts = {}
total_images = 0
for cls in classes:
    cls_path = os.path.join(DATA_ROOT, cls)
    count = len([f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])
    class_counts[cls] = count
    total_images += count

print(f"Total images: {total_images}")
print(f"Average images per class: {total_images/len(classes):.1f}")
print(f"Min: {min(class_counts.values())} | Max: {max(class_counts.values())}")

In [ ]:
import os
import random
import copy
import time
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import Counter
import timm
import warnings
warnings.filterwarnings('ignore')

# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Setup complete! Device: {device}")

In [ ]:
# Build full dataset list
image_paths = []
image_labels = []

valid_ext = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp')
for cls in classes:
    cls_path = os.path.join(DATA_ROOT, cls)
    for img_name in os.listdir(cls_path):
        if img_name.lower().endswith(valid_ext):
            image_paths.append(os.path.join(cls_path, img_name))
            image_labels.append(cls)

# Encode labels
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(image_labels)
num_classes = len(label_encoder.classes_)

print(f"✅ Dataset prepared:")
print(f"   Total images: {len(image_paths)}")
print(f"   Classes: {num_classes}")

In [ ]:
# 70/15/15 split with stratification
X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths, encoded_labels,
    test_size=0.30, stratify=encoded_labels, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50, stratify=y_temp, random_state=42
)

print(f"✅ Data split:")
print(f"   Train: {len(X_train)}")
print(f"   Val:   {len(X_val)}")
print(f"   Test:  {len(X_test)}")

In [ ]:
IMG_SIZE = 224

# IMPROVED TRAINING TRANSFORMS - tuned for ancient script
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
    transforms.RandomCrop(IMG_SIZE),

    # NO horizontal flip (would mirror letters!)

    # Small rotation (rocks aren't tilted much)
    transforms.RandomRotation(degrees=8),

    # Perspective for 3D rock surfaces
    transforms.RandomPerspective(distortion_scale=0.15, p=0.3),

    # Affine for position/scale variations
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.92, 1.08),
        shear=3,
    ),

    # Strong color augmentation (rock lighting)
    transforms.ColorJitter(
        brightness=0.4, contrast=0.4,
        saturation=0.2, hue=0.05,
    ),

    # Slight blur (weathered inscriptions)
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.8)),

    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),

    # Random erasing (damaged inscriptions)
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

print("✅ Script-aware augmentation configured")
print("   ❌ No horizontal flip (preserves letter meaning)")
print("   ✅ Perspective + Affine for rock variations")
print("   ✅ Strong color jitter for lighting changes")
print("   ✅ Random erasing for damaged areas")

In [ ]:
class BrahmiDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
        except Exception as e:
            print(f"Error loading {self.image_paths[idx]}: {e}")
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128, 128, 128))

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(self.labels[idx], dtype=torch.long)


train_dataset = BrahmiDataset(X_train, y_train, transform=train_transform)
val_dataset = BrahmiDataset(X_val, y_val, transform=val_transform)
test_dataset = BrahmiDataset(X_test, y_test, transform=val_transform)

print(f"✅ Datasets created")
print(f"   Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
# Compute weights for imbalanced classes
train_class_counts = Counter(y_train)
total_train = len(y_train)
class_weights_dict = {cls: total_train / count for cls, count in train_class_counts.items()}
sample_weights = [class_weights_dict[label] for label in y_train]
sample_weights = torch.tensor(sample_weights, dtype=torch.float)

weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Create DataLoaders
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    sampler=weighted_sampler, num_workers=2,
    pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)

print(f"✅ DataLoaders ready (batch size: {BATCH_SIZE})")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches:   {len(val_loader)}")
print(f"   Test batches:  {len(test_loader)}")

In [ ]:
gc.collect()
torch.cuda.empty_cache()

# Load pretrained Visformer-Small
model = timm.create_model(
    'visformer_small',
    pretrained=True,
    num_classes=num_classes,
    drop_rate=0.15,
    drop_path_rate=0.15,
)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Pretrained Visformer-Small loaded!")
print(f"   Parameters: {total_params:,}")
print(f"   Pretrained on ImageNet-1K")

# Test forward pass
dummy = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    output = model(dummy)
print(f"   Test output shape: {output.shape}")
print(f"   GPU Memory: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

In [ ]:
# Compute class weights
class_counts = Counter(y_train)
class_weights_tensor = torch.zeros(num_classes)
for cls in range(num_classes):
    count = class_counts.get(cls, 1)
    class_weights_tensor[cls] = total_train / (num_classes * count)

# Normalize
class_weights_tensor = class_weights_tensor / class_weights_tensor.sum() * num_classes
class_weights_tensor = class_weights_tensor.to(device)


class WeightedFocalLoss(nn.Module):
    """Combines class weights, focal loss, and label smoothing."""
    def __init__(self, class_weights, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.class_weights = class_weights
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs, targets,
            weight=self.class_weights,
            reduction='none',
            label_smoothing=self.smoothing
        )
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


criterion = WeightedFocalLoss(
    class_weights=class_weights_tensor,
    gamma=2.0,
    smoothing=0.1
)

print("✅ Loss: Weighted Focal Loss + Label Smoothing")
print(f"   Class weight range: [{class_weights_tensor.min():.2f}, {class_weights_tensor.max():.2f}]")

In [ ]:
def get_param_groups(model, base_lr=1e-4):
    """Different LR for different model parts."""
    head_params = []
    backbone_late = []
    backbone_early = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'head' in name or 'classifier' in name:
            head_params.append(param)
        elif 'stage3' in name or 'norm' in name:
            backbone_late.append(param)
        else:
            backbone_early.append(param)

    return [
        {'params': backbone_early, 'lr': base_lr * 0.05, 'name': 'backbone_early'},
        {'params': backbone_late, 'lr': base_lr * 0.5, 'name': 'backbone_late'},
        {'params': head_params, 'lr': base_lr * 5.0, 'name': 'head'},
    ]


param_groups = get_param_groups(model, base_lr=1e-4)
optimizer = optim.AdamW(param_groups, weight_decay=0.05, betas=(0.9, 0.999))

print("✅ Discriminative learning rates:")
for group in param_groups:
    n = sum(p.numel() for p in group['params'])
    print(f"   {group['name']:<20} LR: {group['lr']:.2e}  ({n:,} params)")


In [ ]:
NUM_EPOCHS = 60
WARMUP_EPOCHS = 5


class WarmupCosineSchedulerMulti:
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr_ratio=0.01):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.min_lr_ratio = min_lr_ratio
        self.base_lrs = [group['lr'] for group in optimizer.param_groups]

    def step(self, epoch):
        if epoch < self.warmup_epochs:
            factor = (epoch + 1) / self.warmup_epochs
        else:
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            factor = self.min_lr_ratio + 0.5 * (1 - self.min_lr_ratio) * (1 + np.cos(np.pi * progress))

        for group, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            group['lr'] = base_lr * factor

        return self.optimizer.param_groups[0]['lr']


scheduler = WarmupCosineSchedulerMulti(optimizer, WARMUP_EPOCHS, NUM_EPOCHS)
print(f"✅ Scheduler: Warmup({WARMUP_EPOCHS}) + Cosine ({NUM_EPOCHS} epochs)")

In [ ]:
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam


def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    index = torch.randperm(x.size(0)).to(x.device)

    _, _, H, W = x.shape
    cut_ratio = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_ratio), int(H * cut_ratio)
    cx, cy = np.random.randint(W), np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    mixed_x = x.clone()
    mixed_x[:, :, bby1:bby2, bbx1:bbx2] = x[index, :, bby1:bby2, bbx1:bbx2]
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (W * H))

    return mixed_x, y, y[index], lam


def mixed_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print("✅ MixUp + CutMix functions ready")

In [ ]:
class ModelEMA:
    """Exponential Moving Average of model weights."""
    def __init__(self, model, decay=0.9999):
        self.module = copy.deepcopy(model)
        self.module.eval()
        self.decay = decay
        for p in self.module.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ema_p, model_p in zip(self.module.parameters(), model.parameters()):
            ema_p.mul_(self.decay).add_(model_p.detach(), alpha=1 - self.decay)
        for ema_b, model_b in zip(self.module.buffers(), model.buffers()):
            ema_b.copy_(model_b)


print("✅ EMA model class defined")

In [ ]:
class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.001, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
            return False

        improved = (score < self.best_score - self.min_delta) if self.mode == 'min' \
                   else (score > self.best_score + self.min_delta)

        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                return True
        return False


early_stopping = EarlyStopping(patience=15, mode='min')
print("✅ Early stopping configured (patience=15)")

In [ ]:
@torch.no_grad()
def validate_amp(model, loader, criterion, device, epoch, phase='Val'):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc=f'Epoch {epoch+1} [{phase}]', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})

    return running_loss / total, 100. * correct / total, all_preds, all_labels


print("✅ Validation function ready")

**TRAINING**

In [ ]:
gc.collect()
torch.cuda.empty_cache()

scaler = GradScaler('cuda')
ema_model = ModelEMA(model, decay=0.9999)

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'val_loss_ema': [], 'val_acc_ema': [],
    'lr': []
}

best_val_acc = 0.0
best_epoch = 0

print("=" * 75)
print(f"🚀 ENHANCED TRAINING - Visformer-Small (Pretrained)")
print(f"=" * 75)
print(f"Improvements: MixUp + CutMix + Focal Loss + EMA + Discriminative LR")
print(f"Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE} | Mixed Precision: ON")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print("=" * 75)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    # Update LR
    current_lr = scheduler.step(epoch)
    history['lr'].append(current_lr)

    # ========== TRAINING ==========
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Train]', leave=False)
    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # Random augmentation choice
        rand = np.random.random()

        with autocast('cuda'):
            if rand < 0.3:  # 30% MixUp
                mixed_images, y_a, y_b, lam = mixup_data(images, labels, alpha=0.2)
                outputs = model(mixed_images)
                loss = mixed_criterion(criterion, outputs, y_a, y_b, lam)
            elif rand < 0.5:  # 20% CutMix
                mixed_images, y_a, y_b, lam = cutmix_data(images, labels, alpha=1.0)
                outputs = model(mixed_images)
                loss = mixed_criterion(criterion, outputs, y_a, y_b, lam)
            else:  # 50% Normal
                outputs = model(images)
                loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        # Update EMA
        ema_model.update(model)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})

    train_loss = running_loss / total
    train_acc = 100. * correct / total

    torch.cuda.empty_cache()

    # ========== VALIDATION ==========
    val_loss, val_acc, _, _ = validate_amp(model, val_loader, criterion, device, epoch, 'Val')
    ema_val_loss, ema_val_acc, _, _ = validate_amp(ema_model.module, val_loader, criterion, device, epoch, 'Val-EMA')

    torch.cuda.empty_cache()

    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_loss_ema'].append(ema_val_loss)
    history['val_acc_ema'].append(ema_val_acc)

    epoch_time = time.time() - epoch_start
    gpu_mem = torch.cuda.max_memory_allocated(0) / 1e9

    print(f"Epoch [{epoch+1:3d}/{NUM_EPOCHS}] "
          f"Train: {train_acc:5.2f}% | "
          f"Val: {val_acc:5.2f}% | "
          f"EMA: {ema_val_acc:5.2f}% | "
          f"LR: {current_lr:.2e} | "
          f"Time: {epoch_time:.1f}s | "
          f"GPU: {gpu_mem:.1f}GB")

    # Save best model (whichever is better - regular or EMA)
    best_current = max(val_acc, ema_val_acc)
    use_ema = ema_val_acc > val_acc
    best_state = ema_model.module.state_dict() if use_ema else model.state_dict()

    if best_current > best_val_acc:
        best_val_acc = best_current
        best_epoch = epoch + 1
        torch.save({
            'epoch': epoch,
            'model_state_dict': best_state,
            'val_acc': float(best_current),
            'num_classes': int(num_classes),
            'label_encoder_classes': list(label_encoder.classes_),
            'used_ema': bool(use_ema),
        }, 'best_model.pth')
        marker = "EMA" if use_ema else "Regular"
        print(f"  ★ NEW BEST: {best_current:.2f}% ({marker} model saved!)")

    torch.cuda.reset_peak_memory_stats()

    if early_stopping(min(val_loss, ema_val_loss)):
        print(f"\n⚠ Early stopping at epoch {epoch+1}")
        break

total_time = time.time() - start_time
print("\n" + "=" * 75)
print(f"🎉 TRAINING COMPLETE!")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Best Epoch: {best_epoch}")
print(f"Best Val Acc: {best_val_acc:.2f}%")
print("=" * 75)

**VISUALIZATION**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', alpha=0.8, color='blue')
axes[0].plot(history['val_loss'], label='Val', alpha=0.8, color='red')
axes[0].plot(history['val_loss_ema'], label='Val-EMA', alpha=0.8, color='green', linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', alpha=0.8, color='blue')
axes[1].plot(history['val_acc'], label='Val', alpha=0.8, color='red')
axes[1].plot(history['val_acc_ema'], label='Val-EMA', alpha=0.8, color='green', linestyle='--')
axes[1].axhline(y=best_val_acc, color='purple', linestyle=':', label=f'Best: {best_val_acc:.2f}%')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate
axes[2].plot(history['lr'], color='purple', alpha=0.8)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Training curves saved!")

 **EVALUATION**

In [ ]:
gc.collect()
torch.cuda.empty_cache()

checkpoint = torch.load('best_model.pth', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"   Type: {'EMA' if checkpoint['used_ema'] else 'Regular'}")
print(f"   Best val acc: {checkpoint['val_acc']:.2f}%")

# Standard test evaluation
test_loss, test_acc, test_preds, test_labels = validate_amp(
    model, test_loader, criterion, device, 0, 'Test'
)

print(f"\n{'='*50}")
print(f"STANDARD TEST RESULTS")
print(f"{'='*50}")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"{'='*50}")

**Test Time Augmentation (TTA) Evaluation**

In [ ]:
@torch.no_grad()
def evaluate_with_tta(model, loader, device):
    """Evaluate using multiple augmented predictions."""
    model.eval()
    correct = 0
    total = 0
    test_preds = []
    test_labels = []

    pbar = tqdm(loader, desc='TTA Test', leave=True)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        all_preds = []

        # Original
        with autocast('cuda'):
            all_preds.append(F.softmax(model(images), dim=1))

        # Rotations
        for angle in [-5, 5, -3, 3]:
            rotated = transforms.functional.rotate(images, angle)
            with autocast('cuda'):
                all_preds.append(F.softmax(model(rotated), dim=1))

        # Translations
        for shift in [(2, 0), (-2, 0), (0, 2)]:
            shifted = transforms.functional.affine(
                images, angle=0, translate=shift, scale=1.0, shear=0
            )
            with autocast('cuda'):
                all_preds.append(F.softmax(model(shifted), dim=1))

        avg_pred = torch.stack(all_preds).mean(dim=0)
        _, predicted = avg_pred.max(1)

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        test_preds.extend(predicted.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({'acc': f'{100.*correct/total:.2f}%'})

    return 100. * correct / total, test_preds, test_labels


print("Running TTA evaluation...")
tta_acc, tta_preds, tta_labels = evaluate_with_tta(model, test_loader, device)

print(f"\n{'='*50}")
print(f"COMPARISON")
print(f"{'='*50}")
print(f"Standard Test Accuracy: {test_acc:.2f}%")
print(f"TTA Test Accuracy:      {tta_acc:.2f}%")
print(f"TTA Improvement:        +{tta_acc - test_acc:.2f}%")
print(f"{'='*50}")

# Use TTA results for final report
final_preds = tta_preds
final_labels = tta_labels
final_acc = tta_acc

**Detailed Classification Report**

In [ ]:
class_names = label_encoder.classes_

report = classification_report(
    final_labels, final_preds,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

print("=" * 70)
print("FINAL CLASSIFICATION REPORT (with TTA)")
print("=" * 70)
print(f"\nOverall Accuracy:        {report['accuracy']*100:.2f}%")
print(f"Macro Avg Precision:     {report['macro avg']['precision']*100:.2f}%")
print(f"Macro Avg Recall:        {report['macro avg']['recall']*100:.2f}%")
print(f"Macro Avg F1-Score:      {report['macro avg']['f1-score']*100:.2f}%")
print(f"Weighted Avg F1-Score:   {report['weighted avg']['f1-score']*100:.2f}%")

# Save full report
full_report = classification_report(
    final_labels, final_preds,
    target_names=class_names,
    zero_division=0
)
with open('classification_report.txt', 'w') as f:
    f.write(full_report)
print("\n✅ Full report saved to 'classification_report.txt'")

# Per-class summary
print("\n" + "=" * 70)
print("PER-CLASS RESULTS (sorted by F1-score)")
print("=" * 70)

per_class_data = []
for cls in class_names:
    if cls in report:
        per_class_data.append({
            'class': cls,
            'precision': report[cls]['precision'] * 100,
            'recall': report[cls]['recall'] * 100,
            'f1': report[cls]['f1-score'] * 100,
            'support': int(report[cls]['support'])
        })

per_class_data.sort(key=lambda x: x['f1'], reverse=True)

print(f"\n🏆 TOP 10 BEST CLASSES:")
for d in per_class_data[:10]:
    print(f"   {d['class']:<15} F1: {d['f1']:.1f}%  P: {d['precision']:.1f}%  R: {d['recall']:.1f}%  N: {d['support']}")

print(f"\n⚠ TOP 10 WORST CLASSES:")
for d in per_class_data[-10:]:
    print(f"   {d['class']:<15} F1: {d['f1']:.1f}%  P: {d['precision']:.1f}%  R: {d['recall']:.1f}%  N: {d['support']}")

**Confusion Matrix**

In [ ]:
cm = confusion_matrix(final_labels, final_preds)

# Full confusion matrix
fig, ax = plt.subplots(figsize=(25, 25))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted', fontsize=14)
ax.set_ylabel('True', fontsize=14)
ax.set_title(f'Confusion Matrix - Test Accuracy: {final_acc:.2f}%', fontsize=16)
plt.xticks(rotation=90, fontsize=5)
plt.yticks(rotation=0, fontsize=5)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

# Top confused pairs
print("\n🔄 TOP 15 MOST CONFUSED CLASS PAIRS:")
print("-" * 60)
confused = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i][j] > 0:
            confused.append((class_names[i], class_names[j], cm[i][j]))

confused.sort(key=lambda x: x[2], reverse=True)
for true_cls, pred_cls, count in confused[:15]:
    print(f"   {true_cls:<15} → predicted as {pred_cls:<15} : {count} times")

**Per-Class Accuracy Plot**

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1) * 100
sorted_idx = np.argsort(per_class_acc)[::-1]

fig, ax = plt.subplots(figsize=(20, 6))
colors = ['green' if a > 90 else 'orange' if a > 70 else 'red'
          for a in per_class_acc[sorted_idx]]

ax.bar(range(len(per_class_acc)), per_class_acc[sorted_idx], color=colors, alpha=0.7)
ax.axhline(y=final_acc, color='blue', linestyle='--', linewidth=2,
           label=f'Overall: {final_acc:.1f}%')
ax.set_xlabel('Class (sorted by accuracy)', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Per-Class Test Accuracy (with TTA)', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
high = sum(1 for a in per_class_acc if a >= 90)
med = sum(1 for a in per_class_acc if 70 <= a < 90)
low = sum(1 for a in per_class_acc if a < 70)
print(f"\n📊 PERFORMANCE SUMMARY:")
print(f"   🟢 Excellent (≥90%): {high} classes")
print(f"   🟡 Good (70-89%):    {med} classes")
print(f"   🔴 Needs work (<70%): {low} classes")

In [ ]:
def predict_image(model, image_path, transform, label_encoder, device, top_k=5, use_tta=True):
    """Predict the class of a single image with optional TTA."""
    model.eval()
    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        if use_tta:
            preds = []
            with autocast('cuda'):
                preds.append(F.softmax(model(input_tensor), dim=1))
            for angle in [-3, 3, -5, 5]:
                rotated = transforms.functional.rotate(input_tensor, angle)
                with autocast('cuda'):
                    preds.append(F.softmax(model(rotated), dim=1))
            probs = torch.stack(preds).mean(dim=0)
        else:
            with autocast('cuda'):
                output = model(input_tensor)
                probs = F.softmax(output, dim=1)

        top_probs, top_indices = torch.topk(probs, top_k)

    results = [(label_encoder.classes_[idx.item()], prob.item() * 100)
               for prob, idx in zip(top_probs[0], top_indices[0])]
    return image, results


# Visualize predictions
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
sample_indices = random.sample(range(len(X_test)), 12)

correct_count = 0
for idx, ax in zip(sample_indices, axes.flatten()):
    image, predictions = predict_image(
        model, X_test[idx], val_transform, label_encoder, device,
        top_k=3, use_tta=True
    )
    true_label = label_encoder.classes_[y_test[idx]]
    is_correct = predictions[0][0] == true_label
    if is_correct:
        correct_count += 1

    ax.imshow(image)
    pred_text = f"True: {true_label}\n"
    for cls, prob in predictions:
        marker = "✓" if cls == true_label else "✗"
        pred_text += f"{marker} {cls}: {prob:.1f}%\n"

    color = 'green' if is_correct else 'red'
    ax.set_title(pred_text, fontsize=8, color=color, loc='left')
    ax.axis('off')

plt.suptitle(f'Sample Predictions (TTA) - {correct_count}/12 Correct',
             fontsize=16)
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ Sample predictions: {correct_count}/12 correct ({100*correct_count/12:.0f}%)")


**Upload New Image and Predict**

In [ ]:
# Cell 31: Upload your own image and get predictions
from google.colab import files
import torch
import torch.nn.functional as F
from torch.amp import autocast
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os

# ============================================================
# STEP 1: Upload your image
# ============================================================
print("📤 Please upload your Brahmi letter image...")
print("   (Supported: .jpg, .jpeg, .png, .bmp)")
print()

uploaded = files.upload()

# Get the uploaded filename
if not uploaded:
    raise Exception("No file uploaded!")

image_filename = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {image_filename}")


# ============================================================
# STEP 2: Load the trained model (if not already loaded)
# ============================================================
import gc
gc.collect()
torch.cuda.empty_cache()

# Check if model is already loaded, otherwise load it
try:
    model
    print("✅ Using existing model in memory")
except NameError:
    print("Loading model from saved file...")
    import timm

    checkpoint = torch.load('best_model.pth', weights_only=False)
    num_classes = checkpoint['num_classes']

    model = timm.create_model(
        'visformer_small',
        pretrained=False,
        num_classes=num_classes,
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    print(f"✅ Model loaded from checkpoint (epoch {checkpoint['epoch']+1})")

model.eval()


# ============================================================
# STEP 3: Define the prediction function
# ============================================================
def predict_uploaded_image(image_path, model, label_encoder, device, top_k=10, use_tta=True):
    """
    Predict the Brahmi letter in an uploaded image.
    Returns top-k predictions with confidence percentages.
    """
    # Define preprocessing (same as training validation transform)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)

    # Make prediction
    with torch.no_grad():
        if use_tta:
            # Test Time Augmentation - average multiple predictions
            all_preds = []

            # Original image
            with autocast('cuda'):
                all_preds.append(F.softmax(model(input_tensor), dim=1))

            # Slight rotations
            for angle in [-5, -3, 3, 5]:
                rotated = transforms.functional.rotate(input_tensor, angle)
                with autocast('cuda'):
                    all_preds.append(F.softmax(model(rotated), dim=1))

            # Slight translations
            for shift in [(2, 0), (-2, 0), (0, 2), (0, -2)]:
                shifted = transforms.functional.affine(
                    input_tensor, angle=0, translate=shift, scale=1.0, shear=0
                )
                with autocast('cuda'):
                    all_preds.append(F.softmax(model(shifted), dim=1))

            # Average all predictions
            probs = torch.stack(all_preds).mean(dim=0)
        else:
            with autocast('cuda'):
                output = model(input_tensor)
                probs = F.softmax(output, dim=1)

        # Get top K predictions
        top_probs, top_indices = torch.topk(probs, top_k)

    # Convert to readable format
    results = []
    for prob, idx in zip(top_probs[0], top_indices[0]):
        class_name = label_encoder.classes_[idx.item()]
        confidence = prob.item() * 100
        results.append((class_name, confidence))

    return image, results


# ============================================================
# STEP 4: Make the prediction
# ============================================================
print("\n🔍 Analyzing image with Test Time Augmentation...")
print("   (averaging 9 predictions for higher accuracy)\n")

image, predictions = predict_uploaded_image(
    image_path=image_filename,
    model=model,
    label_encoder=label_encoder,
    device=device,
    top_k=10,
    use_tta=True
)


# ============================================================
# STEP 5: Display results - VISUAL
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# LEFT: Show the uploaded image
axes[0].imshow(image)
axes[0].set_title(f'Uploaded Image\n({image_filename})', fontsize=14, fontweight='bold')
axes[0].axis('off')

# RIGHT: Show prediction bar chart (top 10)
top_n = 10
class_names = [p[0] for p in predictions[:top_n]]
confidences = [p[1] for p in predictions[:top_n]]

# Color: green for top, fading to gray for lower ones
colors = ['#2ecc71' if i == 0 else
          '#3498db' if i < 3 else
          '#95a5a6' for i in range(top_n)]

bars = axes[1].barh(range(top_n), confidences, color=colors, alpha=0.8, edgecolor='black')
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(class_names, fontsize=11)
axes[1].invert_yaxis()  # Highest at top
axes[1].set_xlabel('Confidence (%)', fontsize=12)
axes[1].set_title(f'Top {top_n} Predictions', fontsize=14, fontweight='bold')
axes[1].set_xlim(0, 100)
axes[1].grid(True, alpha=0.3, axis='x')

# Add percentage labels on bars
for i, (bar, conf) in enumerate(zip(bars, confidences)):
    axes[1].text(conf + 1, bar.get_y() + bar.get_height()/2,
                 f'{conf:.2f}%',
                 ha='left', va='center',
                 fontweight='bold' if i == 0 else 'normal',
                 fontsize=11)

plt.tight_layout()
plt.savefig('my_prediction.png', dpi=150, bbox_inches='tight')
plt.show()


# ============================================================
# STEP 6: Display results - TEXT (detailed)
# ============================================================
print("\n" + "=" * 70)
print("🎯 PREDICTION RESULTS")
print("=" * 70)
print(f"\n📸 Image: {image_filename}")
print(f"📐 Size: {image.size[0]} × {image.size[1]} pixels")
print(f"\n🏆 MOST LIKELY LETTER:")
print(f"   ┌─────────────────────────────────────┐")
print(f"   │  Class: {predictions[0][0]:<20}        │")
print(f"   │  Confidence: {predictions[0][1]:.2f}%              │")
print(f"   └─────────────────────────────────────┘")

# Confidence interpretation
top_conf = predictions[0][1]
if top_conf >= 90:
    confidence_msg = "🟢 VERY HIGH CONFIDENCE - Model is very certain"
elif top_conf >= 75:
    confidence_msg = "🟢 HIGH CONFIDENCE - Model is confident"
elif top_conf >= 50:
    confidence_msg = "🟡 MEDIUM CONFIDENCE - Model is fairly sure"
elif top_conf >= 30:
    confidence_msg = "🟠 LOW CONFIDENCE - Multiple possibilities"
else:
    confidence_msg = "🔴 VERY LOW CONFIDENCE - Image might be unclear"

print(f"\n{confidence_msg}")

print(f"\n📊 TOP 10 PREDICTIONS:")
print(f"   {'Rank':<6} {'Class':<20} {'Confidence':<12} {'Bar'}")
print(f"   {'-'*6} {'-'*20} {'-'*12} {'-'*30}")
for i, (cls, conf) in enumerate(predictions[:10], 1):
    bar_length = int(conf / 100 * 30)
    bar = '█' * bar_length + '░' * (30 - bar_length)
    rank_marker = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f" {i}"
    print(f"   {rank_marker:<6} {cls:<20} {conf:>6.2f}%      {bar}")

print("\n" + "=" * 70)


# ============================================================
# STEP 7: Additional analysis
# ============================================================
# Calculate prediction certainty (entropy)
probs_np = np.array([p[1]/100 for p in predictions])
probs_np = probs_np / probs_np.sum()  # Normalize
entropy = -np.sum(probs_np * np.log(probs_np + 1e-10))
max_entropy = np.log(len(probs_np))
certainty = (1 - entropy/max_entropy) * 100

print(f"\n📈 PREDICTION ANALYSIS:")
print(f"   • Certainty Score: {certainty:.1f}%")
print(f"   • Top-1 vs Top-2 gap: {predictions[0][1] - predictions[1][1]:.2f}%")

if predictions[0][1] - predictions[1][1] < 10:
    print(f"   ⚠ WARNING: Top 2 predictions are very close!")
    print(f"     Consider these as possible matches:")
    print(f"     - {predictions[0][0]} ({predictions[0][1]:.1f}%)")
    print(f"     - {predictions[1][0]} ({predictions[1][1]:.1f}%)")

print("\n✅ Analysis complete!")
print(f"📁 Results saved to: my_prediction.png")

**FINAL SUMMARY**

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║         BRAHMI LETTER RECOGNITION - FINAL SUMMARY            ║
╠══════════════════════════════════════════════════════════════╣
""")
print(f"║  📊 Dataset: {num_classes} classes, {total_images} total images")
print(f"║  🏗  Model: Pretrained Visformer-Small")
print(f"║")
print(f"║  📈 RESULTS:")
print(f"║     Standard Test Accuracy:  {test_acc:.2f}%")
print(f"║     TTA Test Accuracy:       {tta_acc:.2f}%")
print(f"║     Macro F1-Score:          {report['macro avg']['f1-score']*100:.2f}%")
print(f"║     Best Validation Acc:     {best_val_acc:.2f}%")
print(f"║")
print(f"║  ⚡ TECHNIQUES APPLIED:")
print(f"║     ✓ Pretrained ImageNet weights")
print(f"║     ✓ Discriminative learning rates")
print(f"║     ✓ Weighted Focal Loss + Label Smoothing")
print(f"║     ✓ MixUp + CutMix augmentation")
print(f"║     ✓ EMA (Exponential Moving Average)")
print(f"║     ✓ Test Time Augmentation (TTA)")
print(f"║     ✓ Mixed Precision Training (AMP)")
print(f"║     ✓ Script-aware augmentation (no flip)")
print(f"║")
print(f"║  🎓 FOR YOUR PAPER:")
print(f"║     Mention all techniques as your contribution")
print(f"║     Compare with baseline (no improvements): ~79%")
print(f"║     Final achievement: {tta_acc:.2f}%")
print(f"║     Improvement: +{tta_acc - 79.55:.2f}%")
print(f"║")
print("╚══════════════════════════════════════════════════════════════╝")

**Final Wording**

We propose an enhanced Visformer-Small architecture for Early Brahmi inscription recognition. Our approach leverages transfer learning with ImageNet pretrained weights and incorporates several state-of-the-art techniques: (1) discriminative learning rates with backbone trained at 5×10⁻⁶ and classification head at 5×10⁻⁴; (2) Weighted Focal Loss (γ=2.0) combined with label smoothing (α=0.1) to address severe class imbalance; (3) MixUp (α=0.2) and CutMix (α=1.0) augmentation strategies applied stochastically; (4) Exponential Moving Average (EMA, decay=0.9999) of model weights; (5) Test-Time Augmentation (TTA) with rotational and translational variants during inference; and (6) script-aware data augmentation that preserves character identity by avoiding horizontal flipping. Our model achieves [your_acc]% test accuracy on 110 Brahmi character classes, representing a [improvement]% absolute improvement over the baseline.